# Day 016：Pretrain 数据处理

本 Notebook 用最小例子复现 `PretrainDataset.__getitem__()` 的关键逻辑。它不依赖本地的大型预训练 JSONL 文件，适合先理解 BOS、EOS、PAD、labels 和 next-token 对齐。

## 1. 预训练样本的目标

预训练数据通常只有 `text` 字段。程序把正文 token 变成 `BOS + 正文 + EOS`，再补 PAD 到固定长度。labels 先复制 input_ids，PAD 的 label 改为 `-100`。

In [ ]:
import torch
import torch.nn.functional as F

max_length = 8
bos_id, eos_id, pad_id = 1, 2, 0
body_tokens = [10, 20, 30]

tokens = [bos_id] + body_tokens[:max_length - 2] + [eos_id]
input_ids = tokens + [pad_id] * (max_length - len(tokens))
input_ids = torch.tensor(input_ids, dtype=torch.long)

labels = input_ids.clone()
labels[input_ids == pad_id] = -100

print('input_ids:', input_ids)
print('labels:   ', labels)

## 2. 为什么正文长度是 `max_length - 2`

BOS 和 EOS 各占一个位置。如果正文最多取 `max_length` 个 token，再加两个特殊 token 就会超长。因此源码使用 `max_length - 2`。

In [ ]:
long_body = list(range(100, 600))
reserved_body = long_body[:max_length - 2]
long_tokens = [bos_id] + reserved_body + [eos_id]
print('正文原长度:', len(long_body))
print('保留正文长度:', len(reserved_body))
print('加 BOS/EOS 后:', len(long_tokens))
assert len(long_tokens) == max_length

## 3. next-token 对齐

模型位置 `t` 的 logits 预测位置 `t+1` 的 token。MiniMind 的 `forward()` 使用 `logits[..., :-1]` 和 `labels[..., 1:]` 完成错位。

In [ ]:
x_positions = input_ids[:-1]
y_targets = labels[1:]
print('模型看到的输入位置:', x_positions.tolist())
print('对应目标位置:      ', y_targets.tolist())
print('配对:', list(zip(x_positions.tolist(), y_targets.tolist())))

在这个例子中，训练目标是 `10、20、30、EOS`。第一个 BOS 不作为目标；PAD 的目标是 `-100`，交叉熵会忽略它们。

In [ ]:
vocab_size = 40
logits = torch.randn(input_ids.shape[0] - 1, vocab_size)
loss = F.cross_entropy(
    logits,
    y_targets,
    ignore_index=-100
)
valid_count = (y_targets != -100).sum()
print('有效目标数量:', valid_count.item())
print('loss:', loss.item())

## 4. Pretrain 与 Full SFT

两者共享 `MiniMindForCausalLM`、next-token 交叉熵和训练循环。区别主要在数据格式、起始权重和 labels 的监督位置：Pretrain 几乎监督所有正文 token；SFT 主要监督 assistant 回答 token。